### Name: Ashish Mulani
### PRN: 260240128007
------------------------------
### Name: Pranita Lad
### PRN: 260240128028

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import  ColumnTransformer, make_column_selector
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import log_loss
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [3]:
# train = pd.read_csv('C:/Users/PGCP-AI/Downloads/IrrigationKaggle/train.csv')
train = pd.read_csv('D:/download chrome/Irrigation/train.csv', index_col=0)
train

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,629995,Clay,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,...,Rice,Sowing,Kharif,Sprinkler,River,4.35,No,118.36,South,Medium
629996,629996,Clay,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,...,Sugarcane,Vegetative,Kharif,Drip,Groundwater,12.97,Yes,40.75,Central,Medium
629997,629997,Clay,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,...,Potato,Vegetative,Zaid,Canal,Reservoir,13.58,Yes,2.62,South,High
629998,629998,Clay,5.93,42.86,0.33,3.39,34.99,94.58,2433.92,9.88,...,Sugarcane,Vegetative,Zaid,Sprinkler,Groundwater,0.79,Yes,90.00,East,Low


In [5]:
test = pd.read_csv('D:\download chrome\Irrigation/test.csv')
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [6]:
le = LabelEncoder()
train['Irrigation_Need'] = le.fit_transform(train['Irrigation_Need'])

X = train.drop('Irrigation_Need',axis=1)
y=train['Irrigation_Need']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26, stratify=train['Irrigation_Need'])

In [7]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(
    transformers=[("OHE", ohe, make_column_selector(dtype_include=object))], remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

scaler = StandardScaler()
X_trn_scl = scaler.fit_transform(X_trn_ohe)
X_tst_scl = scaler.transform(X_tst_ohe)

In [10]:
# Logistic Regression

solvers = ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga']  # Liblinear not works for huge multiclass dataset.
Cs = np.linspace(0.001, 5, 10)
scores = []
for s in tqdm(solvers):
    for c in Cs:
        lr=LogisticRegression(solver=s, C=c)  
        lr.fit(X_trn_scl, y_train)
        y_pred_prob = lr.predict_proba(X_tst_scl)
        scores.append([s, c, log_loss(y_test, y_pred_prob)])
        
df_scores = pd.DataFrame(scores, columns=['solver', 'C', 'Log_Loss'])
df_scores.sort_values('Log_Loss', ascending=True)

100%|███████████████████████████████████████████████████████████████████████████████████| 5/5 [10:21<00:00, 124.34s/it]


,solver,C,Log_Loss
20,newton-cholesky,0.001000,0.319110
21,newton-cholesky,0.556444,0.308975
22,newton-cholesky,1.111889,0.308968
23,newton-cholesky,1.667333,0.308966
24,newton-cholesky,2.222778,0.308964
25,newton-cholesky,2.778222,0.308964
26,newton-cholesky,3.333667,0.308963
27,newton-cholesky,3.889111,0.308963
28,newton-cholesky,4.444556,0.308963
29,newton-cholesky,5.000000,0.308962


In [11]:
# KNN 

Ks=[1,2,3,4,5]
scores=[]

for k in tqdm(Ks):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_trn_scl, y_train)
    y_pred_prob = knn.predict_proba(X_tst_scl)
    scores.append([k, log_loss(y_test, y_pred_prob)])
    
df_scores = pd.DataFrame(scores, columns=['Ks', 'Log_loss score'])
df_scores.sort_values('Log_loss score', ascending=True).head()

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [07:07<00:00, 85.58s/it]


,Ks,Log_loss score
4,5,1.160248
3,4,1.548164
2,3,2.268576
1,2,3.952637
0,1,8.869790


In [12]:
# Naive Bayes

nb = GaussianNB()
nb.fit(X_trn_scl,y_train) 
y_pred_prob = nb.predict_proba(X_tst_scl) 
print("Log Loss =", log_loss(y_test, y_pred_prob))

Log Loss = 0.48752666338093603


# Inferencing

In [23]:
X_ohe = trans.fit_transform(X)
X_scl = scaler.fit_transform(X_ohe)
bm = LogisticRegression(solver = 'newton-cholesky', C = 0.001)
bm.fit(X_scl, y)

LogisticRegression(C=0.31911, solver='newton-cholesky')

In [24]:
test_ohe = trans.transform(test)
test_scl = scaler.transform(test_ohe)

In [30]:
submit = pd.read_csv("D:/download chrome/Irrigation/sample_submission.csv")
y_pred = bm.predict(test_scl)

y_pred_labels = le.inverse_transform(y_pred)

submit['Irrigation_Need'] = y_pred_labels


In [31]:
submit

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Low
269996,899996,Low
269997,899997,Medium
269998,899998,Low


In [33]:
submit.to_csv("D:/download chrome/Irrigation/sbt_log_22_Apr.csv", index=False)